# 04 — Temporal / Sequence Feature Engineering

## Purpose
Prepare event sequences and regularized weekly panels for temporal analysis, plus a one-row-per-student feature table suitable for a future XGBoost/Gradient Boosting classifier.

### Important target constraint
The dataset gives each student one eventual `dropped_out` label, but no dropout timestamp. We can create chronological activity sequences and use the student-level label as an outcome association, but **we cannot assign a trustworthy week-level dropout target or train a true time-to-dropout/hazard model**. The weekly panel therefore carries the label only as contextual metadata; do not treat each student-week as an independent labeled example.

### Model suitability
- **XGBoost / Gradient Boosting:** can use student-level temporal summaries and lag/rolling features; use a student-level split.
- **ARIMA:** designed for forecasting a numeric, regularly sampled series, not directly for classifying a binary student-level dropout label. We can prepare weekly activity series for exploratory forecasting, but the absence of dropout timestamps prevents direct ARIMA dropout-event prediction.
- **Sequence models:** possible in principle, but 500 students and relatively short/irregular histories are limited data. Establish simple baselines first.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

ROOT_CANDIDATES = [Path("."), Path(".."), Path("/mnt/data/student_success_assistant_v2")]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / "data" / "students.csv").exists()), Path(".").resolve())
DATA = ROOT / "data"
CURATED = DATA / "curated"
CURATED.mkdir(parents=True, exist_ok=True)

students_raw = pd.read_csv(DATA / "students.csv")
attempts_raw = pd.read_csv(DATA / "quiz_attempts.csv")
outcomes_raw = pd.read_csv(DATA / "outcome.csv")

students = students_raw.copy()
attempts = attempts_raw.copy()
outcomes = outcomes_raw.copy()
students["signup_dt"] = pd.to_datetime(students["signup_date"], errors="coerce")
attempts["attempt_dt"] = pd.to_datetime(attempts["attempted_at"], errors="coerce")
attempts["score_num"] = pd.to_numeric(attempts["score"], errors="coerce")
attempts["duration_num"] = pd.to_numeric(attempts["time_spent_min"], errors="coerce")
attempts["score_valid"] = attempts["score_num"].between(0, 100, inclusive="both")
attempts["duration_valid"] = attempts["duration_num"].ge(0)
attempts = attempts.merge(students[["student_id", "signup_dt"]], on="student_id", how="left", validate="many_to_one")
attempts["days_from_signup"] = (attempts["attempt_dt"].dt.normalize() - attempts["signup_dt"].dt.normalize()).dt.days
outcomes["dropped_out"] = outcomes["dropped_out"].astype("string").str.strip().str.lower()
outcomes["target_dropout"] = outcomes["dropped_out"].map({"yes": 1, "no": 0})

assert students["student_id"].is_unique
assert outcomes["student_id"].is_unique and outcomes["target_dropout"].notna().all()
assert attempts["attempt_id"].is_unique
assert set(attempts["student_id"].dropna()).issubset(set(students["student_id"]))

COURSE_START = pd.Timestamp("2026-06-01")
COURSE_END = pd.Timestamp("2026-09-30 23:59:59")
print("Project root:", ROOT)
print("Students:", len(students), "| Attempts:", len(attempts))
print("Parsed timestamps:", int(attempts["attempt_dt"].notna().sum()), "/", len(attempts))
print("Invalid scores:", int((~attempts["score_valid"]).sum()))
print("Pre-signup attempts:", int((attempts["days_from_signup"] < 0).sum()))

Project root: /Users/abishek/Documents/Projects/student_success_assistant
Students: 500 | Attempts: 2453
Parsed timestamps: 2453 / 2453
Invalid scores: 3
Pre-signup attempts: 0


## 1. Build the chronological attempt sequence

In [2]:
# Keep each attempt as an event. Invalid scores remain visible but are represented as missing valid_score.
seq = attempts.copy()
seq["valid_score"] = seq["score_num"].where(seq["score_valid"])
seq["valid_duration_min"] = seq["duration_num"].where(seq["duration_valid"])
seq["event_time_valid"] = seq["attempt_dt"].notna().astype("int8")
seq = seq.sort_values(["student_id", "attempt_dt", "attempt_id"], na_position="last").reset_index(drop=True)

g = seq.groupby("student_id", sort=False)
seq["sequence_index"] = g.cumcount() + 1
seq["previous_attempt_at"] = g["attempt_dt"].shift(1)
seq["gap_since_previous_attempt_days"] = (
    (seq["attempt_dt"] - seq["previous_attempt_at"]).dt.total_seconds() / 86400
)
seq["previous_valid_score"] = g["valid_score"].shift(1)
# Shift first so current attempt does not leak into the "past-only" rolling features.
seq["prior_valid_score_count"] = g["valid_score"].transform(lambda s: s.shift(1).notna().cumsum())
seq["prior_score_mean_3"] = g["valid_score"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
seq["prior_score_std_3"] = g["valid_score"].transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
seq["prior_score_mean_all"] = g["valid_score"].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
# Count distinct lesson IDs observed before the current event (past-only).
seq["prior_lesson_count"] = 0
for _, idx in seq.groupby("student_id", sort=False).groups.items():
    seen = set()
    for row_idx in idx:
        seq.loc[row_idx, "prior_lesson_count"] = len(seen)
        lesson = seq.loc[row_idx, "lesson_id"]
        if pd.notna(lesson):
            seen.add(lesson)
seq["days_from_signup"] = seq["days_from_signup"].where(seq["days_from_signup"] >= 0)

seq = seq.merge(outcomes[["student_id", "target_dropout"]], on="student_id", how="left", validate="many_to_one")
# Target is attached for grouping/visualization only. It is not a per-event future label.
seq["target_semantics"] = "student_level_outcome_no_dropout_timestamp"

seq_cols = [
    "attempt_id", "student_id", "lesson_id", "attempt_dt", "signup_dt", "days_from_signup",
    "sequence_index", "gap_since_previous_attempt_days", "score_num", "valid_score",
    "score_valid", "score_invalid_flag", "duration_num", "valid_duration_min",
    "duration_valid", "previous_valid_score", "prior_valid_score_count",
    "prior_score_mean_3", "prior_score_std_3", "prior_score_mean_all",
    "prior_lesson_count", "target_dropout", "target_semantics"
]
# Include a robust invalid-score flag if not already present.
seq["score_invalid_flag"] = (~seq["score_valid"]).astype("int8")
seq["duration_missing_flag"] = seq["duration_num"].isna().astype("int8")
seq["duration_negative_flag"] = (seq["duration_num"] < 0).fillna(False).astype("int8")
seq_cols += ["duration_missing_flag", "duration_negative_flag"]
attempt_sequence = seq[seq_cols].copy()

print("Sequence rows:", len(attempt_sequence), "| unique students:", attempt_sequence.student_id.nunique())
display(attempt_sequence.head(12))
attempt_sequence.to_csv(CURATED / "temporal_attempt_sequences.csv", index=False)


Sequence rows: 2453 | unique students: 499


,attempt_id,student_id,lesson_id,attempt_dt,signup_dt,days_from_signup,sequence_index,gap_since_previous_attempt_days,score_num,valid_score,...,previous_valid_score,prior_valid_score_count,prior_score_mean_3,prior_score_std_3,prior_score_mean_all,prior_lesson_count,target_dropout,target_semantics,duration_missing_flag,duration_negative_flag
0,A1028,S001,L1,2026-07-18 08:58:00,2026-07-18,0,1,NaN,92,92.0,...,NaN,0,NaN,NaN,NaN,0,0,student_level_outcome_no_dropout_timestamp,1,0
1,A2288,S001,L1,2026-07-22 14:01:00,2026-07-18,4,2,4.210417,86,86.0,...,92.0,1,92.000000,NaN,92.000000,1,0,student_level_outcome_no_dropout_timestamp,0,0
2,A1729,S001,L1,2026-07-27 20:16:00,2026-07-18,9,3,5.260417,82,82.0,...,86.0,2,89.000000,4.242641,89.000000,1,0,student_level_outcome_no_dropout_timestamp,0,0
3,A2125,S002,L1,2026-07-17 01:33:00,2026-07-11,6,1,NaN,77,77.0,...,NaN,0,NaN,NaN,NaN,0,0,student_level_outcome_no_dropout_timestamp,0,0
4,A1504,S002,L1,2026-07-26 14:31:00,2026-07-11,15,2,9.540278,59,59.0,...,77.0,1,77.000000,NaN,77.000000,1,0,student_level_outcome_no_dropout_timestamp,0,0
5,A1599,S002,L2,2026-07-28 22:25:00,2026-07-11,17,3,2.329167,69,69.0,...,59.0,2,68.000000,12.727922,68.000000,1,0,student_level_outcome_no_dropout_timestamp,0,0
6,A1216,S002,L2,2026-08-11 05:03:00,2026-07-11,31,4,13.276389,64,64.0,...,69.0,3,68.333333,9.018500,68.333333,2,0,student_level_outcome_no_dropout_timestamp,0,0
7,A1508,S002,L2,2026-08-13 05:09:00,2026-07-11,33,5,2.004167,63,63.0,...,64.0,4,64.000000,5.000000,67.250000,2,0,student_level_outcome_no_dropout_timestamp,1,0
8,A1776,S002,L3,2026-08-21 02:12:00,2026-07-11,41,6,7.877083,49,49.0,...,63.0,5,65.333333,3.214550,66.400000,2,0,student_level_outcome_no_dropout_timestamp,0,0
9,A0667,S002,L3,2026-08-23 08:36:00,2026-07-11,43,7,2.266667,43,43.0,...,49.0,6,58.666667,8.386497,63.500000,3,0,student_level_outcome_no_dropout_timestamp,0,0


### Sequence table interpretation

Each row is an attempt event sorted within student. `prior_score_mean_3`, `prior_score_mean_all`, `previous_valid_score`, and prior lesson count are computed from earlier rows only. Invalid scores remain in the sequence but are not treated as valid score observations.

**Do not randomly split this event table by row.** The same student appears repeatedly. If it is later used in a supervised experiment, split by `student_id` so a student's events cannot appear in both train and test. More importantly, because the label has no event time, this table does not provide valid week-by-week dropout targets.


## 2. Create a regular student-week panel

In [3]:
# Weekly bins use course calendar weeks, beginning Monday. Generate a panel from each signup week
# through the course end, including weeks with no attempts. This makes inactivity explicit.
def week_start(ts):
    return ts.normalize() - pd.Timedelta(days=ts.weekday())

student_rows = []
for _, s in students.iterrows():
    if pd.isna(s["signup_dt"]):
        continue
    first_week = max(week_start(s["signup_dt"]), week_start(COURSE_START))
    weeks = pd.date_range(first_week, week_start(COURSE_END), freq="7D")
    for w in weeks:
        student_rows.append({
            "student_id": s["student_id"],
            "week_start": w,
            "signup_dt": s["signup_dt"],
            "week_index_from_signup": int((w - week_start(s["signup_dt"])).days // 7),
            "age_group": s["age_group"],
            "country": s["country"],
            "device": s["device"],
        })
panel = pd.DataFrame(student_rows)

event_rows = attempts[
    attempts["attempt_dt"].notna()
    & attempts["signup_dt"].notna()
    & (attempts["attempt_dt"] >= attempts["signup_dt"])
    & (attempts["attempt_dt"] <= COURSE_END)
].copy()
event_rows["week_start"] = event_rows["attempt_dt"].map(week_start)
event_rows["valid_score"] = event_rows["score_num"].where(event_rows["score_valid"])
event_rows["valid_duration_min"] = event_rows["duration_num"].where(event_rows["duration_valid"])

weekly_agg = event_rows.groupby(["student_id", "week_start"], as_index=False).agg(
    attempts_this_week=("attempt_id", "count"),
    valid_scores_this_week=("valid_score", "count"),
    mean_score_this_week=("valid_score", "mean"),
    median_score_this_week=("valid_score", "median"),
    score_std_this_week=("valid_score", "std"),
    invalid_scores_this_week=("score_valid", lambda s: int((~s).sum())),
    mean_duration_this_week=("valid_duration_min", "mean"),
    missing_duration_attempts_this_week=("duration_num", lambda s: int(s.isna().sum())),
    lessons_attempted_this_week=("lesson_id", "nunique"),
    active_days_this_week=("attempt_dt", lambda s: int(s.dt.normalize().nunique())),
    first_attempt_this_week=("attempt_dt", "min"),
    last_attempt_this_week=("attempt_dt", "max"),
)
weekly_panel = panel.merge(weekly_agg, on=["student_id", "week_start"], how="left", validate="one_to_one")
count_cols = ["attempts_this_week", "valid_scores_this_week", "invalid_scores_this_week",
              "missing_duration_attempts_this_week", "lessons_attempted_this_week", "active_days_this_week"]
weekly_panel[count_cols] = weekly_panel[count_cols].fillna(0).astype(int)
weekly_panel = weekly_panel.merge(outcomes[["student_id", "target_dropout"]], on="student_id", how="left", validate="many_to_one")
weekly_panel["target_semantics"] = "student_level_outcome_no_dropout_timestamp"
weekly_panel = weekly_panel.sort_values(["student_id", "week_start"]).reset_index(drop=True)

# Past/cumulative activity features; per-student cumulative sums are safe as sequence descriptors.
weekly_panel["cumulative_attempts"] = weekly_panel.groupby("student_id")["attempts_this_week"].cumsum()
weekly_panel["cumulative_valid_scores"] = weekly_panel.groupby("student_id")["valid_scores_this_week"].cumsum()
weekly_panel["cumulative_mean_score"] = (
    weekly_panel.groupby("student_id")["mean_score_this_week"]
    .transform(lambda s: s.expanding(min_periods=1).mean())
)
weekly_panel["weeks_since_signup"] = weekly_panel["week_index_from_signup"]
weekly_panel["no_activity_this_week"] = (weekly_panel["attempts_this_week"] == 0).astype("int8")

print("Weekly panel rows:", len(weekly_panel), "| students:", weekly_panel.student_id.nunique())
display(weekly_panel.head(15))
weekly_panel.to_csv(CURATED / "temporal_student_weekly_panel.csv", index=False)


Weekly panel rows: 7058 | students: 500


,student_id,week_start,signup_dt,week_index_from_signup,age_group,country,device,attempts_this_week,valid_scores_this_week,mean_score_this_week,...,active_days_this_week,first_attempt_this_week,last_attempt_this_week,target_dropout,target_semantics,cumulative_attempts,cumulative_valid_scores,cumulative_mean_score,weeks_since_signup,no_activity_this_week
0,S001,2026-07-13,2026-07-18,0,35-44,UAE,mobile,1,1,92.0,...,1,2026-07-18 08:58:00,2026-07-18 08:58:00,0,student_level_outcome_no_dropout_timestamp,1,1,92.000000,0,0
1,S001,2026-07-20,2026-07-18,1,35-44,UAE,mobile,1,1,86.0,...,1,2026-07-22 14:01:00,2026-07-22 14:01:00,0,student_level_outcome_no_dropout_timestamp,2,2,89.000000,1,0
2,S001,2026-07-27,2026-07-18,2,35-44,UAE,mobile,1,1,82.0,...,1,2026-07-27 20:16:00,2026-07-27 20:16:00,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,2,0
3,S001,2026-08-03,2026-07-18,3,35-44,UAE,mobile,0,0,NaN,...,0,NaT,NaT,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,3,1
4,S001,2026-08-10,2026-07-18,4,35-44,UAE,mobile,0,0,NaN,...,0,NaT,NaT,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,4,1
5,S001,2026-08-17,2026-07-18,5,35-44,UAE,mobile,0,0,NaN,...,0,NaT,NaT,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,5,1
6,S001,2026-08-24,2026-07-18,6,35-44,UAE,mobile,0,0,NaN,...,0,NaT,NaT,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,6,1
7,S001,2026-08-31,2026-07-18,7,35-44,UAE,mobile,0,0,NaN,...,0,NaT,NaT,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,7,1
8,S001,2026-09-07,2026-07-18,8,35-44,UAE,mobile,0,0,NaN,...,0,NaT,NaT,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,8,1
9,S001,2026-09-14,2026-07-18,9,35-44,UAE,mobile,0,0,NaN,...,0,NaT,NaT,0,student_level_outcome_no_dropout_timestamp,3,3,86.666667,9,1


### How to use the weekly panel safely

This table is a **temporal representation**, not a ready-made independent-row training dataset. The `target_dropout` column repeats the final student-level outcome for context. Since the actual dropout time is unknown, a row for (say) week 2 cannot be labelled as “will drop out after week 2” with confidence.

For a future dynamic prediction system, obtain a dropout/last-active definition and a label horizon (for example, no activity for 30 days after a weekly index date). Without that definition, use the panel for exploratory plots and derive one row per student for a fairer first supervised comparison.


## 3. Derive one-row-per-student temporal summary features

In [4]:
# These summarize each student's attempt timeline. This is the recommended temporal-feature
# dataset for XGBoost/Gradient Boosting against the supplied student-level outcome.
def build_temporal_summary(srow):
    sid = srow["student_id"]
    d = event_rows[event_rows["student_id"].eq(sid)].sort_values(["attempt_dt", "attempt_id"]).copy()
    valid = d[d["score_valid"] & d["score_num"].notna()].copy()
    ts = d["attempt_dt"].dropna().sort_values()
    gaps = ts.diff().dt.total_seconds().dropna() / 86400
    row = {
        "student_id": sid,
        "target_dropout": int(outcomes.loc[outcomes["student_id"].eq(sid), "target_dropout"].iloc[0]),
        "age_group": srow["age_group"],
        "country": srow["country"],
        "device": srow["device"],
        "signup_weekday": int(srow["signup_dt"].dayofweek) if pd.notna(srow["signup_dt"]) else np.nan,
        "attempt_count": int(len(d)),
        "valid_score_count": int(len(valid)),
        "invalid_score_count": int((~d["score_valid"]).sum()),
        "distinct_lessons": int(d["lesson_id"].nunique()),
        "active_days": int(ts.dt.normalize().nunique()) if len(ts) else 0,
        "active_weeks": int(ts.map(week_start).nunique()) if len(ts) else 0,
        "first_attempt_delay_days": float((ts.min() - srow["signup_dt"]).total_seconds()/86400) if len(ts) and pd.notna(srow["signup_dt"]) else np.nan,
        "last_attempt_day_from_signup": float((ts.max() - srow["signup_dt"]).total_seconds()/86400) if len(ts) and pd.notna(srow["signup_dt"]) else np.nan,
        "span_first_to_last_attempt_days": float((ts.max() - ts.min()).total_seconds()/86400) if len(ts) > 1 else 0.0 if len(ts) == 1 else np.nan,
        "mean_inter_attempt_gap_days": float(gaps.mean()) if len(gaps) else np.nan,
        "median_inter_attempt_gap_days": float(gaps.median()) if len(gaps) else np.nan,
        "max_inter_attempt_gap_days": float(gaps.max()) if len(gaps) else np.nan,
        "std_inter_attempt_gap_days": float(gaps.std()) if len(gaps) > 1 else np.nan,
        "mean_score": float(valid["score_num"].mean()) if len(valid) else np.nan,
        "median_score": float(valid["score_num"].median()) if len(valid) else np.nan,
        "score_std": float(valid["score_num"].std()) if len(valid) > 1 else np.nan,
        "score_min": float(valid["score_num"].min()) if len(valid) else np.nan,
        "score_max": float(valid["score_num"].max()) if len(valid) else np.nan,
        "first_valid_score": float(valid.sort_values("attempt_dt")["score_num"].iloc[0]) if len(valid) else np.nan,
        "last_valid_score": float(valid.sort_values("attempt_dt")["score_num"].iloc[-1]) if len(valid) else np.nan,
        "score_delta_first_to_last": float(valid.sort_values("attempt_dt")["score_num"].iloc[-1] - valid.sort_values("attempt_dt")["score_num"].iloc[0]) if len(valid) >= 2 else np.nan,
        "score_trend_points_per_day": np.nan,
        "mean_duration_min": float(d.loc[d["duration_valid"], "duration_num"].mean()) if d["duration_valid"].any() else np.nan,
        "duration_missing_count": int(d["duration_num"].isna().sum()),
        "duration_negative_count": int((d["duration_num"] < 0).sum()),
        "no_attempts_flag": int(len(d) == 0),
        "no_valid_scores_flag": int(len(valid) == 0),
    }
    if len(valid) >= 2:
        x = (valid["attempt_dt"] - valid["attempt_dt"].min()).dt.total_seconds().to_numpy()/86400
        y = valid["score_num"].to_numpy(dtype=float)
        if np.unique(x).size > 1:
            row["score_trend_points_per_day"] = float(np.polyfit(x, y, 1)[0])
    for lesson in ["L1", "L2", "L3"]:
        dl = valid[valid["lesson_id"].astype("string").eq(lesson)].sort_values("attempt_dt")
        row[f"{lesson.lower()}_attempt_count"] = int((d["lesson_id"].astype("string").eq(lesson)).sum())
        row[f"{lesson.lower()}_mean_score"] = float(dl["score_num"].mean()) if len(dl) else np.nan
        row[f"{lesson.lower()}_latest_score"] = float(dl["score_num"].iloc[-1]) if len(dl) else np.nan
    return row

temporal_summary = pd.DataFrame([build_temporal_summary(s) for _, s in students.iterrows()])
assert temporal_summary["student_id"].is_unique
assert len(temporal_summary) == len(students)
assert temporal_summary["target_dropout"].notna().all()
display(temporal_summary.head())
print("Temporal summary shape:", temporal_summary.shape)
temporal_summary.to_csv(CURATED / "temporal_student_summary.csv", index=False)


,student_id,target_dropout,age_group,country,device,signup_weekday,attempt_count,valid_score_count,invalid_score_count,distinct_lessons,...,no_valid_scores_flag,l1_attempt_count,l1_mean_score,l1_latest_score,l2_attempt_count,l2_mean_score,l2_latest_score,l3_attempt_count,l3_mean_score,l3_latest_score
0,S001,0,35-44,UAE,mobile,5,3,3,0,1,...,0,3,86.666667,82.0,0,NaN,NaN,0,NaN,NaN
1,S002,0,18-24,Sri Lanka,mobile,5,7,7,0,3,...,0,2,68.000000,59.0,3,65.333333,63.0,2,46.0,43.0
2,S003,0,35-44,Sri Lanka,mobile,0,6,6,0,3,...,0,1,66.000000,66.0,2,74.000000,70.0,3,73.0,57.0
3,S004,0,25-34,India,mobile,2,3,3,0,2,...,0,1,58.000000,58.0,0,NaN,NaN,2,62.5,66.0
4,S005,0,18-24,India,desktop,0,7,7,0,3,...,0,2,98.500000,100.0,2,47.000000,45.0,3,59.0,56.0


Temporal summary shape: (500, 42)


## 4. Data integrity checks, manifests and ARIMA/XGBoost guidance

In [5]:
# Integrity checks
assert attempt_sequence["attempt_id"].is_unique
assert set(attempt_sequence["student_id"]).issubset(set(students["student_id"]))
assert weekly_panel.groupby(["student_id", "week_start"]).size().max() == 1
assert temporal_summary["student_id"].is_unique
assert temporal_summary["target_dropout"].isin([0, 1]).all()

checks = pd.DataFrame([
    {"dataset": "temporal_attempt_sequences.csv", "rows": len(attempt_sequence), "unique_students": attempt_sequence.student_id.nunique(), "unique_key": attempt_sequence.attempt_id.is_unique},
    {"dataset": "temporal_student_weekly_panel.csv", "rows": len(weekly_panel), "unique_students": weekly_panel.student_id.nunique(), "unique_key": not weekly_panel.duplicated(["student_id", "week_start"]).any()},
    {"dataset": "temporal_student_summary.csv", "rows": len(temporal_summary), "unique_students": temporal_summary.student_id.nunique(), "unique_key": temporal_summary.student_id.is_unique},
])
display(checks)
checks.to_csv(CURATED / "temporal_dataset_verification.csv", index=False)

manifest = pd.DataFrame([
    {"file": "temporal_attempt_sequences.csv", "grain": "one row per attempt", "best_use": "Sequence inspection, event-level rolling features, exploratory plots", "caution": "Repeated student-level label is context only; do not row-split or treat as per-event target"},
    {"file": "temporal_student_weekly_panel.csv", "grain": "one row per student-week", "best_use": "Activity timeline plots and future dynamic feature research", "caution": "Repeated student-level label is not a valid week-specific dropout target"},
    {"file": "temporal_student_summary.csv", "grain": "one row per student", "best_use": "XGBoost/Gradient Boosting with temporal summary features", "caution": "All observed history is used; not an early snapshot unless features are rebuilt at a cutoff"},
])
display(manifest)
manifest.to_csv(CURATED / "temporal_dataset_manifest.csv", index=False)

print("Curated temporal datasets saved to:", CURATED.resolve())


,dataset,rows,unique_students,unique_key
0,temporal_attempt_sequences.csv,2453,499,True
1,temporal_student_weekly_panel.csv,7058,500,True
2,temporal_student_summary.csv,500,500,True


,file,grain,best_use,caution
0,temporal_attempt_sequences.csv,one row per attempt,"Sequence inspection, event-level rolling featu...",Repeated student-level label is context only; ...
1,temporal_student_weekly_panel.csv,one row per student-week,Activity timeline plots and future dynamic fea...,Repeated student-level label is not a valid we...
2,temporal_student_summary.csv,one row per student,XGBoost/Gradient Boosting with temporal summar...,All observed history is used; not an early sna...


Curated temporal datasets saved to: /Users/abishek/Documents/Projects/student_success_assistant/data/curated


## Recommendation for the modeling stage

1. **Primary model comparison:** train baseline + Random Forest + Gradient Boosting/XGBoost on `classification_snapshot_14d.csv`.
2. **Temporal feature comparison:** train Gradient Boosting/XGBoost on `temporal_student_summary.csv`, but explicitly call this an all-observed-history comparison, not a 14-day early-warning model.
3. **Do not directly train ARIMA to classify dropout.** It could forecast a numeric series such as weekly attempt counts or mean score, but it needs a sufficiently regular series and still would not tell us when dropout happened. For a defensible dynamic dropout model, define dropout operationally (for example, no activity for a specified number of days) and create horizon-specific labels.
4. Use a student-level split in all experiments. Never split attempt or student-week rows randomly across train/test.
5. Because the dataset is small (500 students), report uncertainty and avoid overclaiming generalization.
